# 02 Expression data preparation (GTEx + TCGA)

**Purpose** \
 Build, from the UCSC Xena *TCGA TARGET GTEx* cohort and the frozen KEGG tables of notebook 01, the two datasets used by every downstream script, with **identical genes, identical pathways and identical preprocessing rules** so that the GTEx and TCGA models are comparable.

**Inputs**
- `TcgaTargetGtex_rsem_gene_tpm.gz`: 60,498 genes × 19,131 samples, values are `log2(TPM + 0.001)` (floor −9.9658 = non-expressed).
- `TcgaTargetGTEX_phenotype.txt`: sample annotations.
- `data/kegg/`: output of notebook 01.

**Outputs** (`data/prepared/`)
- `gtex.npz`, `tcga.npz`: `X` (samples × genes, float32, log2 scale, **not** standardised), `y`, `classes`, `sample_ids`, `train_idx`, `test_idx`.
- `meta.json`: gene table (ensembl / entrez / symbol / column index), `pathway_map` (name → column indices), all selection parameters and counts.
- `study_effect_pca.png`, `bin_diagnostics.tsv`.

**Pipeline**
1. Read only the KEGG-mapped genes from the matrix (chunked, ~10 min, < 2 GB RAM).
2. Select samples and classes: GTEx *Normal Tissue*, TCGA *Primary Tumor*, classes with ≥ 100 samples (GTEx brain regions collapsed).
3. Stratified train/test split per dataset (fixed here, reused by every script).
4. Expression filter on the union of the two **training** sets (unsupervised): a gene is kept if `log2(TPM+0.001) ≥ EXPR_MIN` in at least `MIN_FRACTION` of training samples.
5. Resolve Entrez IDs with several Ensembl IDs (keep the one with highest mean expression).
6. Pathway size criterion [10, 150] on the surviving genes → final pathway set.
7. Diagnostics: effective number of quantile bins per gene; PCA study effect.

In [45]:
import os, json, time, gzip, collections
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# ---------------- configuration ----------------
EXPR_FILE   = "TcgaTargetGtex_rsem_gene_tpm.gz"
PHENO_FILE  = "TcgaTargetGTEX_phenotype.txt.gz"
KEGG_DIR    = "data/kegg"
#OUT_DIR     = "data/prepared"
OUT_DIR     = "data/prepared_200"

MIN_SAMPLES_PER_CLASS = 100      # classes kept in each dataset
TEST_SIZE   = 0.25               # stratified split, same seed as Biomni's pilots
SPLIT_SEED  = 42
EXPR_MIN    = 0.0                # log2(TPM+0.001) >= 0  <=>  TPM >= ~1
MIN_FRACTION = 0.10              # ... in at least 10% of training samples (union GTEx+TCGA)
SIZE_MIN = 10                    # pathway size criterion on surviving genes
#SIZE_MAX = 150                  # pathway size criterion on surviving genes
SIZE_MAX = 200                   # pathway size criterion on surviving genes
N_BINS      = 6                  # for the bin diagnostic only (the model fits its own on train)
FLOOR       = -9.9658            # log2(0.001)
SAMPLE_MAX_FLOOR = 0.50          # sample QC: drop samples with > 50% of candidate genes at the floor

os.makedirs(OUT_DIR, exist_ok=True)

## 1. Read the KEGG-mapped genes from the expression matrix

The matrix has genes as rows (versioned Ensembl IDs) and samples as columns. Reading it whole needs ~9 GB (float64); instead we stream it in chunks of 2,000 rows and keep only rows whose unversioned Ensembl ID belongs to a KEGG pathway that passed criterion 1. Candidate genes are taken **before** the size criterion, so thresholds can be changed later without re-reading the file.

In [30]:
pw = pd.read_csv(f"{KEGG_DIR}/kegg_pathways.tsv", sep="\t")
pw_genes = pd.read_csv(f"{KEGG_DIR}/kegg_pathway_genes.tsv", sep="\t")
gene_map = pd.read_csv(f"{KEGG_DIR}/gene_map.tsv", sep="\t")

kept_pw = set(pw.loc[pw["pass_criterion1"], "pid"])
cand_entrez = set(pw_genes.loc[pw_genes["pid"].isin(kept_pw), "entrez"])
cand_ens = set(gene_map.loc[gene_map["entrez"].isin(cand_entrez), "ensembl"])
print(f"criterion-1 pathways: {len(kept_pw)} | candidate Entrez: {len(cand_entrez)} | candidate Ensembl IDs: {len(cand_ens)}")

t0 = time.time()
chunks = []
header = pd.read_csv(EXPR_FILE, sep="\t", nrows=0, compression="gzip").columns
reader = pd.read_csv(EXPR_FILE, sep="\t", index_col=0, chunksize=2000, compression="gzip",
                     dtype={c: np.float32 for c in header[1:]})
for i, ch in enumerate(reader):
    ens = ch.index.str.split(".").str[0]
    keep = ens.isin(cand_ens)
    if keep.any():
        sub = ch.loc[keep].copy()
        sub.index = ens[keep]
        chunks.append(sub)
    if i % 5 == 0:
        print(f"  chunk {i:2d}: {sum(len(c) for c in chunks):5d} genes kept so far ({time.time() - t0:.0f}s)")
expr = pd.concat(chunks)                     # genes x samples
del chunks
# GENCODE has a few genes annotated twice (chrX and its PAR_Y copy share the ENSG): keep the higher-expressed row
dup = expr.index.duplicated(keep=False)
if dup.any():
    order = expr.mean(axis=1).values
    expr = expr.iloc[np.lexsort((-order, expr.index.values))]
    expr = expr[~expr.index.duplicated(keep="first")]
print(f"\nread {expr.shape[0]} genes x {expr.shape[1]} samples in {time.time() - t0:.0f}s, "
      f"{expr.values.nbytes / 1e9:.2f} GB, duplicated Ensembl rows resolved: {int(dup.sum() // 2)}")
print(f"candidate Ensembl IDs not found in the matrix: {len(cand_ens - set(expr.index))}")

criterion-1 pathways: 267 | candidate Entrez: 8880 | candidate Ensembl IDs: 10000
  chunk  0:   274 genes kept so far (13s)
  chunk  5:  1647 genes kept so far (65s)
  chunk 10:  2974 genes kept so far (120s)
  chunk 15:  4417 genes kept so far (184s)
  chunk 20:  5762 genes kept so far (244s)
  chunk 25:  7156 genes kept so far (300s)
  chunk 30:  8274 genes kept so far (352s)

read 8274 genes x 19131 samples in 353s, 0.63 GB, duplicated Ensembl rows resolved: 0
candidate Ensembl IDs not found in the matrix: 1726


**Sample QC**\
The fraction of genes at the floor (−9.9658, i.e. TPM = 0) per sample is bimodal: a compact bulk between 0.02 and 0.42 (mode 0.13; the upper tail is low-complexity tissues such as whole blood) and an isolated group at 0.97–1.00, i.e. failed libraries (25 GTEx samples from a single sequencing plate, SM-5CH*, and TCGA-25-1870-01). The threshold 0.5 falls in the empty gap, so its exact value is immaterial. QC is applied before any class count.

In [46]:
import matplotlib.pyplot as plt
frac_floor_sample = (expr <= FLOOR + 1e-4).mean(axis=0)          # one value per sample
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].hist(frac_floor_sample, bins=100); ax[0].set_yscale("log")
ax[0].set_xlabel("fraction of genes at floor per sample"); ax[0].set_ylabel("samples (log)")
ax[1].hist(frac_floor_sample[frac_floor_sample < 0.3], bins=60)
ax[1].set_xlabel("same, zoom on the bulk (< 0.3)")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/floor_fraction_hist.png", dpi=150)

print(f"median {frac_floor_sample.median():.3f} | 99th pct {frac_floor_sample.quantile(0.99):.3f} | "
      f"max of the bulk {frac_floor_sample[frac_floor_sample < 0.5].max():.3f} | "
      f"min of the tail {frac_floor_sample[frac_floor_sample >= 0.5].min():.3f} | "
      f"n >= 0.5: {(frac_floor_sample >= 0.5).sum()}")

median 0.132 | 99th pct 0.228 | max of the bulk 0.417 | min of the tail nan | n >= 0.5: 0


In [47]:
bad_samples = frac_floor_sample[frac_floor_sample > SAMPLE_MAX_FLOOR].index
print(f"samples removed by QC (> {SAMPLE_MAX_FLOOR:.0%} at floor): {len(bad_samples)}")
expr = expr.drop(columns=bad_samples)
print("remaining samples:", expr.shape[1])

samples removed by QC (> 50% at floor): 0
remaining samples: 19103


## 2. Samples and classes

GTEx: `_sample_type == "Normal Tissue"` (this drops the EBV-transformed lymphocytes and cultured fibroblasts, which are cell lines). Brain sub-regions (13 `detailed_category` values with `_primary_site == "Brain"`) are collapsed into one class, otherwise they would dominate the class set with highly similar labels. TCGA: `_sample_type == "Primary Tumor"` only (no metastatic, recurrent, normal-adjacent or blood-derived samples). In both, classes with fewer than `MIN_SAMPLES_PER_CLASS` samples are dropped.

Sample QC (previous section) is applied before counting, so a class that drops below the threshold because of removed samples is excluded (this happens to GTEx Spleen, 100 → 99).

In [48]:
pheno = pd.read_csv(PHENO_FILE, sep="\t", encoding="latin-1").set_index("sample")
pheno = pheno.loc[pheno.index.intersection(expr.columns)]

gtex = pheno[(pheno["_study"] == "GTEX") & (pheno["_sample_type"] == "Normal Tissue")].copy()
gtex["label"] = gtex["detailed_category"]
gtex.loc[gtex["_primary_site"] == "Brain", "label"] = "Brain"

tcga = pheno[(pheno["_study"] == "TCGA") & (pheno["_sample_type"] == "Primary Tumor")].copy()
tcga["label"] = tcga["detailed_category"]

def select_classes(df, name):
    counts = df["label"].value_counts()
    kept = counts[counts >= MIN_SAMPLES_PER_CLASS].index
    out = df[df["label"].isin(kept)].copy()
    print(f"{name}: {len(df)} samples, {len(counts)} classes -> {len(out)} samples, {len(kept)} classes "
          f"(dropped {len(counts) - len(kept)} classes with < {MIN_SAMPLES_PER_CLASS} samples)")
    return out, counts

gtex, gtex_counts = select_classes(gtex, "GTEx")
tcga, tcga_counts = select_classes(tcga, "TCGA")
pd.DataFrame({"GTEx": gtex["label"].value_counts()}).T

GTEx: 7402 samples, 39 classes -> 6855 samples, 28 classes (dropped 11 classes with < 100 samples)
TCGA: 9184 samples, 32 classes -> 8643 samples, 24 classes (dropped 8 classes with < 100 samples)


label,Brain,Muscle - Skeletal,Whole Blood,Skin - Sun Exposed (Lower Leg),Adipose - Subcutaneous,Lung,Artery - Tibial,Nerve - Tibial,Thyroid,Esophagus - Mucosa,...,Testis,Pancreas,Colon - Transverse,Colon - Sigmoid,Esophagus - Gastroesophageal Junction,Adrenal Gland,Artery - Coronary,Liver,Pituitary,Prostate
GTEx,1146,396,337,324,318,287,281,278,278,271,...,165,165,164,140,137,127,118,110,107,100


In [49]:
pd.DataFrame({"TCGA": tcga["label"].value_counts()}).T

label,Breast Invasive Carcinoma,Kidney Clear Cell Carcinoma,Head & Neck Squamous Cell Carcinoma,Lung Adenocarcinoma,Brain Lower Grade Glioma,Thyroid Carcinoma,Lung Squamous Cell Carcinoma,Prostate Adenocarcinoma,Ovarian Serous Cystadenocarcinoma,Stomach Adenocarcinoma,...,Colon Adenocarcinoma,Sarcoma,Esophageal Carcinoma,Uterine Corpus Endometrioid Carcinoma,Pancreatic Adenocarcinoma,Pheochromocytoma & Paraganglioma,Glioblastoma Multiforme,Testicular Germ Cell Tumor,Thymoma,Skin Cutaneous Melanoma
TCGA,1092,530,518,513,509,504,498,495,418,414,...,288,258,181,180,178,177,153,148,119,102


## 3. Train/test split (fixed here, once)

Stratified 75/25 split per dataset with seed 42. Every later script (KG-HTT, baselines, importance) reads these indices instead of splitting again. The expression filter below uses **only training samples**, so the test sets never influence any choice.

In [50]:
def make_split(df):
    classes = sorted(df["label"].unique())
    y = np.array([classes.index(l) for l in df["label"]])
    tr, te = train_test_split(np.arange(len(df)), test_size=TEST_SIZE, stratify=y, random_state=SPLIT_SEED)
    return classes, y, np.sort(tr), np.sort(te)

gtex_classes, gtex_y, gtex_tr, gtex_te = make_split(gtex)
tcga_classes, tcga_y, tcga_tr, tcga_te = make_split(tcga)
print(f"GTEx: train {len(gtex_tr)} / test {len(gtex_te)} | TCGA: train {len(tcga_tr)} / test {len(tcga_te)}")

GTEx: train 5141 / test 1714 | TCGA: train 6482 / test 2161


## 4. Expression filter (unsupervised, training samples only)

A gene is kept if it reaches `EXPR_MIN` (TPM ≈ 1) in at least `MIN_FRACTION` of the pooled training samples of GTEx and TCGA. This is the standard *filterByExpr*-type rule for RNA-seq: genes that are essentially off everywhere carry no signal for either embedding channel and only add parameters. Pooling the two studies guarantees one gene set for both models. The table shows how sensitive the count is to the fraction threshold.

In [51]:
train_samples = np.concatenate([gtex.index[gtex_tr].values, tcga.index[tcga_tr].values])
Xtr = expr[train_samples]                                     # genes x pooled training samples
frac_expr = (Xtr >= EXPR_MIN).mean(axis=1)
frac_floor = (Xtr <= FLOOR + 1e-4).mean(axis=1)

for f in [0.01, 0.05, 0.10, 0.20, 0.50]:
    print(f"  fraction >= {f:.2f}: {int((frac_expr >= f).sum()):5d} genes kept")
keep_genes = frac_expr[frac_expr >= MIN_FRACTION].index
print(f"\nexpression filter (>= {EXPR_MIN} log2TPM in >= {MIN_FRACTION:.0%} of {len(train_samples)} training samples): "
      f"{len(keep_genes)} of {len(expr)} genes kept")
print(f"genes at floor in > 50% of training samples: {(frac_floor > 0.5).sum()} before, "
      f"{(frac_floor[keep_genes] > 0.5).sum()} after the filter")

  fraction >= 0.01:  7637 genes kept
  fraction >= 0.05:  7230 genes kept
  fraction >= 0.10:  6903 genes kept
  fraction >= 0.20:  6445 genes kept
  fraction >= 0.50:  5598 genes kept

expression filter (>= 0.0 log2TPM in >= 10% of 11623 training samples): 6903 of 8274 genes kept
genes at floor in > 50% of training samples: 1005 before, 45 after the filter


## 5. One Ensembl ID per Entrez ID

Some Entrez IDs map to several Ensembl IDs (alternative haplotypes, e.g. KIR and HLA genes). Among those present in the matrix and surviving the filter, we keep the one with the highest mean training expression; if none survives, the gene is dropped. Each surviving Ensembl ID must also map to a single Entrez ID (otherwise ambiguous, dropped).

In [52]:
gm = gene_map[gene_map["ensembl"].isin(keep_genes)].copy()
gm["mean_expr"] = gm["ensembl"].map(Xtr.mean(axis=1))
gm = gm.sort_values("mean_expr", ascending=False).drop_duplicates("entrez", keep="first")
ambiguous = gm["ensembl"].duplicated(keep=False)
print(f"Ensembl IDs mapping to >1 Entrez (dropped): {gm.loc[ambiguous, 'ensembl'].nunique()}")
gm = gm[~ambiguous].sort_values("ensembl").reset_index(drop=True)
print(f"final one-to-one gene table: {len(gm)} genes")

Ensembl IDs mapping to >1 Entrez (dropped): 0
final one-to-one gene table: 6903 genes


## 6. Pathway size criterion on the surviving genes

Pathway membership is now restricted to the genes that survived. Then `SIZE_MIN ≤ size ≤ SIZE_MAX` is applied. The genes of the final matrix are the union of the genes of the selected pathways (a gene not in any selected pathway would never reach the model).

In [53]:
pg = pw_genes[pw_genes["pid"].isin(kept_pw) & pw_genes["entrez"].isin(gm["entrez"])]
size = pg.groupby("pid").size()
sel = size[(size >= SIZE_MIN) & (size <= SIZE_MAX)].index
pw_sel = pw.set_index("pid").loc[sel]
print(f"pathways with >=1 surviving gene: {len(size)} | after size criterion [{SIZE_MIN}, {SIZE_MAX}]: {len(sel)}")
print(f"  dropped: {(size < SIZE_MIN).sum()} smaller than {SIZE_MIN}, {(size > SIZE_MAX).sum()} larger than {SIZE_MAX}")

final_entrez = sorted(set(pg.loc[pg["pid"].isin(sel), "entrez"]))
genes = gm[gm["entrez"].isin(final_entrez)].reset_index(drop=True)
genes["col"] = np.arange(len(genes))
entrez_to_col = dict(zip(genes["entrez"], genes["col"]))
pathway_map = {pw_sel.loc[p, "name"]: sorted(entrez_to_col[e] for e in pg.loc[pg["pid"] == p, "entrez"]) for p in sel}
sizes = np.array([len(v) for v in pathway_map.values()])
cnt = collections.Counter(c for v in pathway_map.values() for c in v)
print(f"\nFINAL: {len(pathway_map)} pathways, {len(genes)} genes, n_max = {sizes.max()}, median size {int(np.median(sizes))}, "
      f"mean pathways per gene {np.mean(list(cnt.values())):.2f}, genes in >= 5 pathways: {sum(v >= 5 for v in cnt.values())}")
pw_sel.assign(n_final=size[sel]).groupby("category")["n_final"].describe()[["count", "min", "50%", "max"]].astype(int)

pathways with >=1 surviving gene: 267 | after size criterion [10, 200]: 237
  dropped: 16 smaller than 10, 14 larger than 200

FINAL: 237 pathways, 6243 genes, n_max = 197, median size 60, mean pathways per gene 2.71, genes in >= 5 pathways: 729


,count,min,50%,max
category,,,,
Cellular Processes,20,31,118,197
Environmental Information Processing,29,28,121,196
Genetic Information Processing,27,11,44,177
Metabolism,76,10,30,128
Organismal Systems,85,16,80,176


In [54]:
# pathways lost to the upper bound after the expression filter (to report / discuss)
lost = pw.set_index("pid").loc[size[size > SIZE_MAX].index, ["name", "subcategory"]].assign(n_surviving=size[size > SIZE_MAX])
lost.sort_values("n_surviving", ascending=False)

,name,subcategory,n_surviving
pid,,,
hsa04519,Cadherin signaling,Signaling molecules and interaction,343
hsa04151,PI3K-Akt signaling pathway,Signal transduction,315
hsa04010,MAPK signaling pathway,Signal transduction,281
hsa04517,IgSF CAM signaling,Signaling molecules and interaction,281
hsa04144,Endocytosis,Transport and catabolism,245
hsa04142,Lysosome biogenesis,Transport and catabolism,242
hsa04145,Phagocytosis,Transport and catabolism,232
hsa04080,Neuroactive ligand-receptor interaction,Signaling molecules and interaction,228
hsa04060,Cytokine-cytokine receptor interaction,Signaling molecules and interaction,225


## 7. Build the two matrices and save

`X` is stored on the log2 scale, un-standardised: scaling and binning are fitted on the training split inside the training script (so they are part of the model, not of the data).

In [55]:
def build(df, y, classes, tr, te, name):
    X = expr.loc[genes["ensembl"], df.index].values.T.astype(np.float32)     # samples x genes
    np.savez_compressed(f"{OUT_DIR}/{name}.npz", X=X, y=y, classes=np.array(classes),
                        sample_ids=df.index.values.astype(str), train_idx=tr, test_idx=te)
    print(f"{name}: X {X.shape}, {len(classes)} classes, saved")
    return X

X_gtex = build(gtex, gtex_y, gtex_classes, gtex_tr, gtex_te, "gtex")
X_tcga = build(tcga, tcga_y, tcga_classes, tcga_tr, tcga_te, "tcga")

meta = {
    "genes": genes[["col", "ensembl", "entrez", "symbol"]].to_dict(orient="records"),
    "pathway_map": pathway_map,
    "pathway_ids": {pw_sel.loc[p, "name"]: p for p in sel},
    "params": {"MIN_SAMPLES_PER_CLASS": MIN_SAMPLES_PER_CLASS, "TEST_SIZE": TEST_SIZE, "SPLIT_SEED": SPLIT_SEED,
               "EXPR_MIN": EXPR_MIN, "MIN_FRACTION": MIN_FRACTION, "SIZE_MIN": SIZE_MIN, "SIZE_MAX": SIZE_MAX, "SAMPLE_MAX_FLOOR": SAMPLE_MAX_FLOOR},
    "counts": {"candidate_genes_read": int(len(expr)), "genes_after_expression_filter": int(len(keep_genes)),
               "genes_final": int(len(genes)), "pathways_final": int(len(pathway_map)), "n_max": int(sizes.max()), "samples_removed_qc": int(len(bad_samples))},
    "removed_samples": list(bad_samples),
}
json.dump(meta, open(f"{OUT_DIR}/meta.json", "w"), indent=1)
print("meta.json saved")

gtex: X (6855, 6243), 28 classes, saved
tcga: X (8643, 6243), 24 classes, saved
meta.json saved


## 8. Diagnostic: effective number of quantile bins per gene

Same procedure as the model (`KBinsDiscretizer(n_bins=6, strategy="quantile")` fitted on the training split of each dataset). A gene with fewer than 6 effective bins has tied quantile edges, i.e. a large fraction of samples at the floor. This is the check that decides whether a dedicated "non-expressed" bin is needed (option kept in reserve).

In [56]:
import warnings
from sklearn.preprocessing import KBinsDiscretizer

def bin_diagnostic(X, tr, name):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        kbd = KBinsDiscretizer(n_bins=N_BINS, encode="ordinal", strategy="quantile").fit(X[tr])
    n_eff = np.array([len(e) - 1 for e in kbd.bin_edges_])
    return pd.Series(n_eff, index=genes["ensembl"], name=name)

diag = pd.concat([bin_diagnostic(X_gtex, gtex_tr, "gtex_bins"), bin_diagnostic(X_tcga, tcga_tr, "tcga_bins")], axis=1)
diag["frac_expr_train"] = frac_expr.reindex(diag.index).values
diag.to_csv(f"{OUT_DIR}/bin_diagnostics.tsv", sep="\t")
tab = pd.DataFrame({c: diag[c].value_counts().sort_index() for c in ["gtex_bins", "tcga_bins"]}).fillna(0).astype(int)
tab["gtex_%"] = (100 * tab["gtex_bins"] / len(diag)).round(1)
tab["tcga_%"] = (100 * tab["tcga_bins"] / len(diag)).round(1)
tab.index.name = "effective bins"
tab

,gtex_bins,tcga_bins,gtex_%,tcga_%
effective bins,,,,
1,9,2,0.1,0.0
2,17,26,0.3,0.4
3,34,36,0.5,0.6
4,75,80,1.2,1.3
5,189,169,3.0,2.7
6,5919,5930,94.8,95.0


## 9. Diagnostic: study effect (PCA)

PCA on the pooled, standardised matrix (all selected samples of both studies, unsupervised). Two views: coloured by study, and by class within each study. The summary statistic is the distance between the study centroids in PC space divided by the mean within-study spread; values well above 1 mean the study is a dominant axis of variation and justify training separate models.

In [57]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

Xall = np.vstack([X_gtex, X_tcga])
study = np.array(["GTEx"] * len(X_gtex) + ["TCGA"] * len(X_tcga))
labels = np.concatenate([np.array(gtex_classes)[gtex_y], np.array(tcga_classes)[tcga_y]])
pcs = PCA(n_components=10, svd_solver="randomized", random_state=0).fit(StandardScaler().fit_transform(Xall))
Z = pcs.transform(StandardScaler().fit_transform(Xall))
cent = {s: Z[study == s, :2].mean(axis=0) for s in ("GTEx", "TCGA")}
spread = np.mean([np.linalg.norm(Z[study == s, :2] - cent[s], axis=1).mean() for s in ("GTEx", "TCGA")])
ratio = np.linalg.norm(cent["GTEx"] - cent["TCGA"]) / spread
print("explained variance (PC1..5):", np.round(pcs.explained_variance_ratio_[:5], 3))
print(f"study-effect ratio (centroid distance / within-study spread, PC1-2): {ratio:.2f}")

fig, ax = plt.subplots(1, 3, figsize=(18, 5.5))
for s, c in zip(("GTEx", "TCGA"), ("tab:blue", "tab:red")):
    ax[0].scatter(Z[study == s, 0], Z[study == s, 1], s=3, alpha=0.4, c=c, label=s)
ax[0].set_title(f"PC1 vs PC2 by study (ratio {ratio:.2f})"); ax[0].legend(markerscale=5)
for k, s in enumerate(("GTEx", "TCGA"), start=1):
    m = study == s
    for lab in np.unique(labels[m]):
        mm = m & (labels == lab)
        ax[k].scatter(Z[mm, 0], Z[mm, 1], s=3, alpha=0.5, label=lab)
    ax[k].set_title(f"{s}: PC1 vs PC2 by class")
for a in ax:
    a.set_xlabel(f"PC1 ({pcs.explained_variance_ratio_[0]:.1%})"); a.set_ylabel(f"PC2 ({pcs.explained_variance_ratio_[1]:.1%})")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/study_effect_pca.png", dpi=130)
print("saved study_effect_pca.png")

explained variance (PC1..5): [0.265 0.109 0.063 0.044 0.036]
study-effect ratio (centroid distance / within-study spread, PC1-2): 1.35
saved study_effect_pca.png


## 10. Summary

In [58]:
print(json.dumps(meta["counts"], indent=1))
print(f"\nfiles in {OUT_DIR}:", sorted(os.listdir(OUT_DIR)))

{
 "candidate_genes_read": 8274,
 "genes_after_expression_filter": 6903,
 "genes_final": 6243,
 "pathways_final": 237,
 "n_max": 197,
 "samples_removed_qc": 0
}

files in data/prepared_200: ['bin_diagnostics.tsv', 'gtex.npz', 'meta.json', 'study_effect_pca.png', 'tcga.npz']
